<a id="top"></a>
# ✈️ Flight Price Prediction — Data Cleaning & Feature Engineering

**Dataset:** `flight_price.xlsx`  
**Goal:** Clean the raw flight data and convert string columns (dates, times, stops, duration) into numerical features, ready for model building.

<a id="toc"></a>
## 📑 Table of Contents

1. [Import Libraries](#import-libraries)
2. [Load Dataset](#load-dataset)
3. [Initial Data Inspection](#initial-inspection)
   - 3.1 [Missing Values](#missing-values)
   - 3.2 [Duplicate Rows](#duplicates)
   - 3.3 [Dataset Info](#dataset-info)
   - 3.4 [Statistical Summary](#describe)
4. [Feature Engineering](#feature-engineering)
   - 4.1 [Date_of_Journey → Date, Month, Year](#fe-date)
   - 4.2 [Arrival_Time → Arrival_hour, Arrival_min](#fe-arrival)
   - 4.3 [Dep_Time → Departure_hour, Departure_min](#fe-departure)
   - 4.4 [Total_Stops → Numeric](#fe-stops)
   - 4.5 [Additional_Info & Route](#fe-route)
   - 4.6 [Duration → Duration_hour, Duration_min](#fe-duration)
5. [Summary](#summary)

<a id="import-libraries"></a>
## 1. Import Libraries

Pandas and NumPy for data handling, Matplotlib and Seaborn for visualization.

[⬆ Back to Table of Contents](#toc)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

<a id="load-dataset"></a>
## 2. Load Dataset

Read the Excel file into a DataFrame and preview the first rows.

[⬆ Back to Table of Contents](#toc)

In [5]:
df=pd.read_excel('flight_price.xlsx')
df.head(5)

,Airline,Date_of_Journey,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price
0,IndiGo,24/03/2019,Banglore,New Delhi,BLR → DEL,22:20,01:10 22 Mar,2h 50m,non-stop,No info,3897
1,Air India,1/05/2019,Kolkata,Banglore,CCU → IXR → BBI → BLR,05:50,13:15,7h 25m,2 stops,No info,7662
2,Jet Airways,9/06/2019,Delhi,Cochin,DEL → LKO → BOM → COK,09:25,04:25 10 Jun,19h,2 stops,No info,13882
3,IndiGo,12/05/2019,Kolkata,Banglore,CCU → NAG → BLR,18:05,23:30,5h 25m,1 stop,No info,6218
4,IndiGo,01/03/2019,Banglore,New Delhi,BLR → NAG → DEL,16:50,21:35,4h 45m,1 stop,No info,13302


<a id="initial-inspection"></a>
## 3. Initial Data Inspection

Check data quality first: missing values, duplicates, data types and basic statistics.

<a id="missing-values"></a>
### 3.1 Missing Values

Count null values in every column.

[⬆ Back to Table of Contents](#toc)

In [6]:
df.isnull().sum()

Airline            0
Date_of_Journey    0
Source             0
Destination        0
Route              1
Dep_Time           0
Arrival_Time       0
Duration           0
Total_Stops        1
Additional_Info    0
Price              0
dtype: int64

<a id="duplicates"></a>
### 3.2 Duplicate Rows

List fully duplicated rows.

[⬆ Back to Table of Contents](#toc)

In [8]:
df[df.duplicated()]

,Airline,Date_of_Journey,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price
683,Jet Airways,1/06/2019,Delhi,Cochin,DEL → NAG → BOM → COK,14:35,04:25 02 Jun,13h 50m,2 stops,No info,13376
1061,Air India,21/05/2019,Delhi,Cochin,DEL → GOI → BOM → COK,22:00,19:15 22 May,21h 15m,2 stops,No info,10231
1348,Air India,18/05/2019,Delhi,Cochin,DEL → HYD → BOM → COK,17:15,19:15 19 May,26h,2 stops,No info,12392
1418,Jet Airways,6/06/2019,Delhi,Cochin,DEL → JAI → BOM → COK,05:30,04:25 07 Jun,22h 55m,2 stops,In-flight meal not included,10368
1674,IndiGo,24/03/2019,Banglore,New Delhi,BLR → DEL,18:25,21:20,2h 55m,non-stop,No info,7303
...,...,...,...,...,...,...,...,...,...,...,...
10594,Jet Airways,27/06/2019,Delhi,Cochin,DEL → AMD → BOM → COK,23:05,12:35 28 Jun,13h 30m,2 stops,No info,12819
10616,Jet Airways,1/06/2019,Delhi,Cochin,DEL → JAI → BOM → COK,09:40,12:35 02 Jun,26h 55m,2 stops,No info,13014
10634,Jet Airways,6/06/2019,Delhi,Cochin,DEL → JAI → BOM → COK,09:40,12:35 07 Jun,26h 55m,2 stops,In-flight meal not included,11733
10672,Jet Airways,27/06/2019,Delhi,Cochin,DEL → AMD → BOM → COK,23:05,19:00 28 Jun,19h 55m,2 stops,In-flight meal not included,11150


<a id="dataset-info"></a>
### 3.3 Dataset Info

Column dtypes and non-null counts.

[⬆ Back to Table of Contents](#toc)

In [9]:
# get the nfo of the data
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10683 entries, 0 to 10682
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Airline          10683 non-null  object
 1   Date_of_Journey  10683 non-null  object
 2   Source           10683 non-null  object
 3   Destination      10683 non-null  object
 4   Route            10682 non-null  object
 5   Dep_Time         10683 non-null  object
 6   Arrival_Time     10683 non-null  object
 7   Duration         10683 non-null  object
 8   Total_Stops      10682 non-null  object
 9   Additional_Info  10683 non-null  object
 10  Price            10683 non-null  int64 
dtypes: int64(1), object(10)
memory usage: 918.2+ KB


<a id="describe"></a>
### 3.4 Statistical Summary

Summary statistics for numerical columns.

[⬆ Back to Table of Contents](#toc)

In [10]:
df.describe()

,Price
count,10683.000000
mean,9087.064121
std,4611.359167
min,1759.000000
25%,5277.000000
50%,8372.000000
75%,12373.000000
max,79512.000000


<a id="feature-engineering"></a>
## 4. Feature Engineering

Most columns are `object` (string) dtype, so we split them into parts and convert to numeric.

[⬆ Back to Table of Contents](#toc)

In [ ]:
# Feature Engineering Process
df.info() # see here dtype is object mean string convert it into numerical

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10683 entries, 0 to 10682
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Airline          10683 non-null  object
 1   Date_of_Journey  10683 non-null  object
 2   Source           10683 non-null  object
 3   Destination      10683 non-null  object
 4   Route            10682 non-null  object
 5   Dep_Time         10683 non-null  object
 6   Arrival_Time     10683 non-null  object
 7   Duration         10683 non-null  object
 8   Total_Stops      10682 non-null  object
 9   Additional_Info  10683 non-null  object
 10  Price            10683 non-null  int64 
dtypes: int64(1), object(10)
memory usage: 918.2+ KB


<a id="fe-date"></a>
### 4.1 Date_of_Journey → Date, Month, Year

Split `dd/mm/yyyy` into three columns, convert to `int`, then drop the original column.

[⬆ Back to Table of Contents](#toc)

In [12]:
df['Date']=df['Date_of_Journey'].str.split('/').str[0]
df['Month']=df['Date_of_Journey'].str.split('/').str[1]
df['Year']=df['Date_of_Journey'].str.split('/').str[2]

In [ ]:
df.info() # still date , month and year are in str

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10683 entries, 0 to 10682
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Airline          10683 non-null  object
 1   Date_of_Journey  10683 non-null  object
 2   Source           10683 non-null  object
 3   Destination      10683 non-null  object
 4   Route            10682 non-null  object
 5   Dep_Time         10683 non-null  object
 6   Arrival_Time     10683 non-null  object
 7   Duration         10683 non-null  object
 8   Total_Stops      10682 non-null  object
 9   Additional_Info  10683 non-null  object
 10  Price            10683 non-null  int64 
 11  Date             10683 non-null  object
 12  Month            10683 non-null  object
 13  Year             10683 non-null  object
dtypes: int64(1), object(13)
memory usage: 1.1+ MB


In [14]:
df['Date']=df['Date'].astype(int)
df['Month']=df['Month'].astype(int)
df['Year']=df['Year'].astype(int)

In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10683 entries, 0 to 10682
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Airline          10683 non-null  object
 1   Date_of_Journey  10683 non-null  object
 2   Source           10683 non-null  object
 3   Destination      10683 non-null  object
 4   Route            10682 non-null  object
 5   Dep_Time         10683 non-null  object
 6   Arrival_Time     10683 non-null  object
 7   Duration         10683 non-null  object
 8   Total_Stops      10682 non-null  object
 9   Additional_Info  10683 non-null  object
 10  Price            10683 non-null  int64 
 11  Date             10683 non-null  int64 
 12  Month            10683 non-null  int64 
 13  Year             10683 non-null  int64 
dtypes: int64(4), object(10)
memory usage: 1.1+ MB


In [17]:
# now drop the column date of journey
df.drop('Date_of_Journey',axis=1,inplace=True)

In [18]:
df.head(2)

,Airline,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price,Date,Month,Year
0,IndiGo,Banglore,New Delhi,BLR → DEL,22:20,01:10 22 Mar,2h 50m,non-stop,No info,3897,24,3,2019
1,Air India,Kolkata,Banglore,CCU → IXR → BBI → BLR,05:50,13:15,7h 25m,2 stops,No info,7662,1,5,2019


<a id="fe-arrival"></a>
### 4.2 Arrival_Time → Arrival_hour, Arrival_min

Keep only the time part (remove the extra date text), split into hour and minute, convert to `int`, drop the original column.

[⬆ Back to Table of Contents](#toc)

In [20]:
df['Arrival_Time']=df['Arrival_Time'].apply(lambda x:x.split(' ')[0])

In [21]:
df['Arrival_hour']=df['Arrival_Time'].str.split(':').str[0]
df['Arrival_min']=df['Arrival_Time'].str.split(':').str[1]

In [23]:
df['Arrival_hour']=df['Arrival_hour'].astype(int)
df['Arrival_min']=df['Arrival_min'].astype(int)

In [24]:
df.drop('Arrival_Time',axis=1,inplace=True)

In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10683 entries, 0 to 10682
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Airline          10683 non-null  object
 1   Source           10683 non-null  object
 2   Destination      10683 non-null  object
 3   Route            10682 non-null  object
 4   Dep_Time         10683 non-null  object
 5   Duration         10683 non-null  object
 6   Total_Stops      10682 non-null  object
 7   Additional_Info  10683 non-null  object
 8   Price            10683 non-null  int64 
 9   Date             10683 non-null  int64 
 10  Month            10683 non-null  int64 
 11  Year             10683 non-null  int64 
 12  Arrival_hour     10683 non-null  int64 
 13  Arrival_min      10683 non-null  int64 
dtypes: int64(6), object(8)
memory usage: 1.1+ MB


<a id="fe-departure"></a>
### 4.3 Dep_Time → Departure_hour, Departure_min

Same approach as arrival time: split `HH:MM`, convert to `int`, drop `Dep_Time`.

[⬆ Back to Table of Contents](#toc)

In [27]:
df['Departure_hour']=df['Dep_Time'].str.split(':').str[0]
df['Departure_min']=df['Dep_Time'].str.split(':').str[1]
df['Departure_hour']=df['Departure_hour'].astype(int)
df['Departure_min']=df['Departure_min'].astype(int)
df.drop('Dep_Time',axis=1,inplace=True)

In [28]:
df.head(2)

,Airline,Source,Destination,Route,Duration,Total_Stops,Additional_Info,Price,Date,Month,Year,Arrival_hour,Arrival_min,Departure_hour,Departure_min
0,IndiGo,Banglore,New Delhi,BLR → DEL,2h 50m,non-stop,No info,3897,24,3,2019,1,10,22,20
1,Air India,Kolkata,Banglore,CCU → IXR → BBI → BLR,7h 25m,2 stops,No info,7662,1,5,2019,13,15,5,50


<a id="fe-stops"></a>
### 4.4 Total_Stops → Numeric

Fill the missing value with the **mode**, then map text labels (`non-stop`, `1 stop`, ...) to numbers.

[⬆ Back to Table of Contents](#toc)

In [29]:
df['Total_Stops'].unique()

array(['non-stop', '2 stops', '1 stop', '3 stops', nan, '4 stops'],
      dtype=object)

In [30]:
df[df['Total_Stops'].isnull()]

,Airline,Source,Destination,Route,Duration,Total_Stops,Additional_Info,Price,Date,Month,Year,Arrival_hour,Arrival_min,Departure_hour,Departure_min
9039,Air India,Delhi,Cochin,NaN,23h 40m,NaN,No info,7480,6,5,2019,9,25,9,45


In [ ]:
df['Total_Stops'].mode() # most frequent value is 
# one stop nan will replace from this value

0    1 stop
Name: Total_Stops, dtype: object

In [37]:
df['Total_Stops']=df['Total_Stops'].fillna(df['Total_Stops'].mode()[0])

In [38]:
df[df['Total_Stops'].isnull()]

,Airline,Source,Destination,Route,Duration,Total_Stops,Additional_Info,Price,Date,Month,Year,Arrival_hour,Arrival_min,Departure_hour,Departure_min


In [39]:
df['Total_Stops'].unique()

array(['non-stop', '2 stops', '1 stop', '3 stops', '4 stops'],
      dtype=object)

In [40]:
df['Total_Stops']=df['Total_Stops'].map({'non-stop':0, '2 stops':2, '1 stop':1, '3 stops':3, '4 stops':4})

In [41]:
df.head(2)

,Airline,Source,Destination,Route,Duration,Total_Stops,Additional_Info,Price,Date,Month,Year,Arrival_hour,Arrival_min,Departure_hour,Departure_min
0,IndiGo,Banglore,New Delhi,BLR → DEL,2h 50m,0,No info,3897,24,3,2019,1,10,22,20
1,Air India,Kolkata,Banglore,CCU → IXR → BBI → BLR,7h 25m,2,No info,7662,1,5,2019,13,15,5,50


In [42]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10683 entries, 0 to 10682
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Airline          10683 non-null  object
 1   Source           10683 non-null  object
 2   Destination      10683 non-null  object
 3   Route            10682 non-null  object
 4   Duration         10683 non-null  object
 5   Total_Stops      10683 non-null  int64 
 6   Additional_Info  10683 non-null  object
 7   Price            10683 non-null  int64 
 8   Date             10683 non-null  int64 
 9   Month            10683 non-null  int64 
 10  Year             10683 non-null  int64 
 11  Arrival_hour     10683 non-null  int64 
 12  Arrival_min      10683 non-null  int64 
 13  Departure_hour   10683 non-null  int64 
 14  Departure_min    10683 non-null  int64 
dtypes: int64(9), object(6)
memory usage: 1.2+ MB


<a id="fe-route"></a>
### 4.5 Additional_Info & Route

Inspect `Additional_Info`, then drop `Route` (its information is already covered by `Total_Stops`).

[⬆ Back to Table of Contents](#toc)

In [43]:
df['Additional_Info'].unique()

array(['No info', 'In-flight meal not included',
       'No check-in baggage included', '1 Short layover', 'No Info',
       '1 Long layover', 'Change airports', 'Business class',
       'Red-eye flight', '2 Long layover'], dtype=object)

In [44]:
df.drop('Route',axis=1,inplace=True)

<a id="fe-duration"></a>
### 4.6 Duration → Duration_hour, Duration_min

Split values like `2h 50m` into hours and minutes. Missing minutes → `0`; entries with only minutes (e.g. `45m`) get hours = `0`.

[⬆ Back to Table of Contents](#toc)

In [46]:
df['Duration'].head(4)

0    2h 50m
1    7h 25m
2       19h
3    5h 25m
Name: Duration, dtype: object

In [53]:
df['Duration_hour']=df['Duration'].str.split(' ').str[0].str.split('h').str[0]
df['Duration_min']=df['Duration'].str.split(' ').str[1].str.split('m').str[0]

In [56]:
df['Duration_hour'].head(4)

0     2
1     7
2    19
3     5
Name: Duration_hour, dtype: object

In [59]:
df[df['Duration_min'].isnull()]

,Airline,Source,Destination,Duration,Total_Stops,Additional_Info,Price,Date,Month,Year,Arrival_hour,Arrival_min,Departure_hour,Departure_min,Duration_hour,Duration_min
2,Jet Airways,Delhi,Cochin,19h,2,No info,13882,9,6,2019,4,25,9,25,19,NaN
18,Air India,Delhi,Cochin,23h,2,No info,13381,12,6,2019,19,15,20,15,23,NaN
33,Jet Airways,Delhi,Cochin,22h,2,In-flight meal not included,10919,15,6,2019,12,35,14,35,22,NaN
44,Multiple carriers,Delhi,Cochin,12h,1,No info,13062,21,3,2019,21,0,9,0,12,NaN
53,IndiGo,Banglore,Delhi,3h,0,No info,3943,18,6,2019,0,15,21,15,3,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10591,Jet Airways,Delhi,Cochin,23h,2,No info,14300,12,6,2019,19,0,20,0,23,NaN
10638,Jet Airways,Banglore,New Delhi,14h,1,In-flight meal not included,7832,21,3,2019,11,25,21,25,14,NaN
10639,Air India,Delhi,Cochin,38h,3,No info,10493,3,6,2019,19,15,5,15,38,NaN
10673,Jet Airways,Delhi,Cochin,15h,2,No info,16704,27,5,2019,4,25,13,25,15,NaN


In [60]:
df['Duration_min']=df['Duration_min'].fillna(0)

In [ ]:
df[df['Duration_min'].isnull()] # there is no value with nan minutes

,Airline,Source,Destination,Duration,Total_Stops,Additional_Info,Price,Date,Month,Year,Arrival_hour,Arrival_min,Departure_hour,Departure_min,Duration_hour,Duration_min


In [63]:
df['Duration_min']=df['Duration_min'].astype(int)

In [ ]:
df[df['Duration_hour'].isnull()]

,Airline,Source,Destination,Duration,Total_Stops,Additional_Info,Price,Date,Month,Year,Arrival_hour,Arrival_min,Departure_hour,Departure_min,Duration_hour,Duration_min


In [66]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10683 entries, 0 to 10682
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Airline          10683 non-null  object
 1   Source           10683 non-null  object
 2   Destination      10683 non-null  object
 3   Duration         10683 non-null  object
 4   Total_Stops      10683 non-null  int64 
 5   Additional_Info  10683 non-null  object
 6   Price            10683 non-null  int64 
 7   Date             10683 non-null  int64 
 8   Month            10683 non-null  int64 
 9   Year             10683 non-null  int64 
 10  Arrival_hour     10683 non-null  int64 
 11  Arrival_min      10683 non-null  int64 
 12  Departure_hour   10683 non-null  int64 
 13  Departure_min    10683 non-null  int64 
 14  Duration_hour    10683 non-null  object
 15  Duration_min     10683 non-null  int64 
dtypes: int64(10), object(6)
memory usage: 1.3+ MB


In [70]:
# Inspect rows where values aren't pure digits
df[df['Duration_hour'].astype(str).str.contains('m', na=False)][['Duration', 'Duration_hour']]

# Replace values ending with 'm' (or non-digits) with 0, then convert to int
df['Duration_hour'] = pd.to_numeric(df['Duration_hour'], errors='coerce').fillna(0).astype(int)

In [71]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10683 entries, 0 to 10682
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Airline          10683 non-null  object
 1   Source           10683 non-null  object
 2   Destination      10683 non-null  object
 3   Duration         10683 non-null  object
 4   Total_Stops      10683 non-null  int64 
 5   Additional_Info  10683 non-null  object
 6   Price            10683 non-null  int64 
 7   Date             10683 non-null  int64 
 8   Month            10683 non-null  int64 
 9   Year             10683 non-null  int64 
 10  Arrival_hour     10683 non-null  int64 
 11  Arrival_min      10683 non-null  int64 
 12  Departure_hour   10683 non-null  int64 
 13  Departure_min    10683 non-null  int64 
 14  Duration_hour    10683 non-null  int64 
 15  Duration_min     10683 non-null  int64 
dtypes: int64(11), object(5)
memory usage: 1.3+ MB


<a id="summary"></a>
## 5. Summary

- Parsed `Date_of_Journey` into **Date / Month / Year**
- Split `Arrival_Time`, `Dep_Time` and `Duration` into **hour** and **minute** columns
- Filled missing `Total_Stops` with the mode and mapped it to numbers
- Dropped redundant columns: `Date_of_Journey`, `Arrival_Time`, `Dep_Time`, `Route`
- All new features are numeric (`int`) and ready for encoding and model training.

[⬆ Back to Table of Contents](#toc)